In [2]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser  # ✅ Works with your version

# 🔐 Load API keys
import os
from dotenv import load_dotenv
load_dotenv(dotenv_path=".env")
openai_api_key = os.getenv("OPENAI_API_KEY")

prompt = PromptTemplate.from_template("Translate to French: {text}")
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
parser = StrOutputParser()

chain = prompt | llm | parser

response = chain.invoke({"text": "Good Evening"})
print(response)


Bonsoir


In [3]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

prompt = PromptTemplate.from_template("List 5 programming languages, comma-separated.")
parser = CommaSeparatedListOutputParser()
chain = prompt | llm | parser

response = chain.invoke({})
print(response)  # ['Python', 'Java', 'C++', 'JavaScript', 'Ruby']

['Python', 'Java', 'C++', 'JavaScript', 'Ruby']


In [4]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

class ProductInfo(BaseModel):
    name: str = Field(description="Name of the product")
    price: float = Field(description="Price in INR")

parser = PydanticOutputParser(pydantic_object=ProductInfo)

prompt = PromptTemplate(
    template="Extract product name and price from: {text}\n{format_instructions}",
    input_variables=["text"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

chain = prompt | llm | parser

response = chain.invoke({
    "text": "The Redmi  17 is available for ₹27,999."
})
print(response)


name='Redmi 17' price=27999.0


In [10]:

# Change "langchain" to "langchain_core"
from langchain.output_parsers.structured import ResponseSchema
from langchain.output_parsers import StructuredOutputParser


schemas = [
    ResponseSchema(name="company", description="Name of the company"),
    ResponseSchema(name="founder", description="Name of the founder"),
]

parser = StructuredOutputParser.from_response_schemas(schemas)

prompt = PromptTemplate(
    template="Extract company and founder from the text: {text}\n{format_instructions}",
    input_variables=["text"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

chain = prompt | llm | parser

response = chain.invoke({
    "text": "Hope AI was founded by Ramisha Rani in Tamil Nadu."
})
print(response)


ModuleNotFoundError: No module named 'langchain.output_parsers'

In [12]:
import os
from langchain_core.prompts import PromptTemplate
# CORRECT IMPORTS: Both classes must be imported from the structured module
from langchain.output_parsers.structured import ResponseSchema, StructuredOutputParser
from langchain_openai import ChatOpenAI

# 1. Initialize the LLM (Ensure your OPENAI_API_KEY environment variable is set)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 2. Define your schemas
schemas = [
    ResponseSchema(name="company", description="Name of the company"),
    ResponseSchema(name="founder", description="Name of the founder"),
]

parser = StructuredOutputParser.from_response_schemas(schemas)

# 3. Define your prompt
prompt = PromptTemplate(
    template="Extract company and founder from the text: {text}\n{format_instructions}",
    input_variables=["text"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

# 4. Construct the LCEL chain
chain = prompt | llm | parser

# 5. Run the chain
response = chain.invoke({
    "text": "Hope AI was founded by Ramisha Rani in Tamil Nadu."
})

print(response)


ModuleNotFoundError: No module named 'langchain.output_parsers'

In [17]:
import os
from langchain_core.prompts import PromptTemplate
#from langchain_core.pydantic_v1 import BaseModel, Field
from langchain_openai import ChatOpenAI

# 1. Initialize the LLM (Ensure your OPENAI_API_KEY environment variable is set)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 2. Define your desired schema using Pydantic instead of ResponseSchema
class CompanyInfo(BaseModel):
    company: str = Field(description="Name of the company")
    founder: str = Field(description="Name of the founder")

# 3. Bind the schema directly to the model (This replaces StructuredOutputParser completely)
structured_llm = llm.with_structured_output(CompanyInfo)

# 4. Define a clean, simple prompt
prompt = PromptTemplate(
    template="Extract the company and founder details from the text:\n\n{text}",
    input_variables=["text"]
)

# 5. Build your modern LCEL Chain
chain = prompt | structured_llm

# 6. Execute the chain
response = chain.invoke({
    "text": "Hope AI was founded by Ramisha Rani in Tamil Nadu."
})

# The response is instantly a clean, validated Python object!
print("Company:", response.company)
print("Founder:", response.founder)


chain = prompt | llm | parser





Company: Hope AI
Founder: Ramisha Rani
